# Перевод с английского на русский язык с помощью Transformer

В этом ноутбуке мы обучим модель Transformer для перевода текста с английского языка на русский.

## План работы:
1. Подготовка данных
2. Создание токенизатора и словаря
3. Реализация модели Transformer
4. Обучение модели
5. Визуализация потерь
6. Генерация переводов

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import matplotlib.pyplot as plt
import math
import random
import re
from tqdm.auto import tqdm
from collections import Counter
import os
from datasets import load_dataset

# Установка seeds для воспроизводимости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используемое устройство: {DEVICE}')

## 1. Подготовка данных

In [ ]:
# Загрузка данных из Hugging Face datasets
from datasets import load_dataset

# Загружаем датасет OPUS (Helsinki-NLP/opus_books или opus-100)
# OPUS содержит переводы для множества пар языков
# Для английского-русского перевода

print("Загрузка датасета OPUS (en-ru)...")
# Используем opus-100 который больше и содержит субтитры
dataset = load_dataset("Helsinki-NLP/opus-100", "en-ru")
print(f"Загружено {len(dataset['train'])} примеров")
print(f"Валидация: {len(dataset['validation'])} примеров")

# Функция для извлечения и фильтрации данных
def prepare_data(dataset, split='train', max_samples=50000, max_length=30, min_length=2):
    """Подготовка данных для обучения"""
    en_sentences = []
    ru_sentences = []
    
    for i, item in enumerate(dataset[split]):
        if i >= max_samples:
            break
        
        # Получаем текст из полей 'translation'
        if 'translation' in item:
            en_text = item['translation']['en']
            ru_text = item['translation']['ru']
        else:
            # Попытка найти поля автоматически
            keys = list(item.keys())
            en_text = item[keys[0]] if len(keys) > 0 else ""
            ru_text = item[keys[1]] if len(keys) > 1 else ""
        
        # Фильтрация пустых строк
        if not en_text or not ru_text:
            continue
        
        # Очистка текста
        en_text = str(en_text).strip()
        ru_text = str(ru_text).strip()
        
        # Фильтрация по длине (включая токены <SOS> и <EOS>)
        en_len = len(en_text.split())
        ru_len = len(ru_text.split())
        
        if min_length <= en_len < max_length and min_length <= ru_len < max_length:
            en_sentences.append(en_text)
            ru_sentences.append(ru_text)
    
    return en_sentences, ru_sentences

# Загружаем данные
MAX_SAMPLES = 50000  # Можно увеличить для лучшего обучения
MAX_SEQ_LENGTH = 30  # Максимальная длина предложения (без учета <SOS>, <EOS>)
en_sentences, ru_sentences = prepare_data(
    dataset, 
    split='train', 
    max_samples=MAX_SAMPLES, 
    max_length=MAX_SEQ_LENGTH,
    min_length=2
)

print(f'Загружено {len(en_sentences)} пар предложений (отфильтрованы по длине <= {MAX_SEQ_LENGTH})')
print(f'\nПример пары:')
print(f'  EN: {en_sentences[0]}')
print(f'  RU: {ru_sentences[0]}')
print(f'\nПример пары (еще один):')
print(f'  EN: {en_sentences[1]}')
print(f'  RU: {ru_sentences[1]}')

# Проверяем максимальные длины
max_en_len = max(len(s.split()) for s in en_sentences)
max_ru_len = max(len(s.split()) for s in ru_sentences)
print(f'\nМаксимальная длина EN: {max_en_len}')
print(f'Максимальная длина RU: {max_ru_len}')

In [ ]:
# Перемешиваем и разделяем данные на train/val/test
combined = list(zip(en_sentences, ru_sentences))
random.shuffle(combined)
en_sentences, ru_sentences = zip(*combined)

# Разделение на train/val/test (80%/10%/10%)
train_size = int(0.8 * len(en_sentences))
val_size = int(0.1 * len(en_sentences))

en_train = en_sentences[:train_size]
ru_train = ru_sentences[:train_size]

en_val = en_sentences[train_size:train_size + val_size]
ru_val = ru_sentences[train_size:train_size + val_size]

en_test = en_sentences[train_size + val_size:]
ru_test = ru_sentences[train_size + val_size:]

train_data = list(zip(en_train, ru_train))
val_data = list(zip(en_val, ru_val))
test_data = list(zip(en_test, ru_test))

print(f'Размер обучающей выборки: {len(train_data)}')
print(f'Размер валидационной выборки: {len(val_data)}')
print(f'Размер тестовой выборки: {len(test_data)}')
print(f'\nПример из обучающей выборки:')
print(f'  EN: {en_train[0]}')
print(f'  RU: {ru_train[0]}')

## 2. Создание токенизатора и словаря

In [ ]:
class Vocabulary:
    def __init__(self, freq_threshold=1):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold
        
    def __len__(self):
        return len(self.itos)
    
    def build_vocabulary(self, sentences):
        frequencies = Counter()
        
        for sentence in sentences:
            for word in self.tokenize(sentence):
                frequencies[word] += 1
        
        idx = 4
        for word, freq in frequencies.items():
            if freq >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1
    
    def tokenize(self, sentence):
        # Простая токенизация - по словам и пунктуации
        # Добавляем пробелы вокруг пунктуации
        sentence = re.sub(r'([.,!?;:])', r' \1 ', sentence)
        # Разделяем по пробелам и удаляем пустые строки
        tokens = [t.lower() for t in sentence.split() if t.strip()]
        return tokens
    
    def numericalize(self, sentence):
        tokens = self.tokenize(sentence)
        return [self.stoi.get(token, self.stoi["<UNK>"]) for token in tokens]

In [ ]:
# Создаем словари
vocab_en = Vocabulary(freq_threshold=1)
vocab_ru = Vocabulary(freq_threshold=1)

vocab_en.build_vocabulary(en_train)
vocab_ru.build_vocabulary(ru_train)

print(f'Размер словаря EN: {len(vocab_en)}')
print(f'Размер словаря RU: {len(vocab_ru)}')
print(f'\nПример токенов EN: {list(vocab_en.stoi.keys())[:10]}')
print(f'Пример токенов RU: {list(vocab_ru.stoi.keys())[:10]}')

## 3. Dataset и DataLoader

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, en_sentences, ru_sentences, vocab_en, vocab_ru):
        self.en_sentences = en_sentences
        self.ru_sentences = ru_sentences
        self.vocab_en = vocab_en
        self.vocab_ru = vocab_ru
    
    def __len__(self):
        return len(self.en_sentences)
    
    def __getitem__(self, idx):
        en_sentence = self.en_sentences[idx]
        ru_sentence = self.ru_sentences[idx]
        
        # Конвертируем в индексы
        en_indices = self.vocab_en.numericalize(en_sentence)
        ru_indices = self.vocab_ru.numericalize(ru_sentence)
        
        # Добавляем токены <SOS> и <EOS>
        en_indices = [self.vocab_en.stoi["<SOS>"]] + en_indices + [self.vocab_en.stoi["<EOS>"]]
        ru_indices = [self.vocab_ru.stoi["<SOS>"]] + ru_indices + [self.vocab_ru.stoi["<EOS>"]]
        
        return {
            'en': torch.tensor(en_indices, dtype=torch.long),
            'ru': torch.tensor(ru_indices, dtype=torch.long)
        }

# Функция для паддинга батчей
def collate_fn(batch):
    en_batch = [item['en'] for item in batch]
    ru_batch = [item['ru'] for item in batch]
    
    en_padded = pad_sequence(en_batch, batch_first=True, padding_value=vocab_en.stoi["<PAD>"])
    ru_padded = pad_sequence(ru_batch, batch_first=True, padding_value=vocab_ru.stoi["<PAD>"])
    
    return {
        'en': en_padded,
        'ru': ru_padded
    }

# Создаем датасеты
train_dataset = TranslationDataset(en_train, ru_train, vocab_en, vocab_ru)
val_dataset = TranslationDataset(en_val, ru_val, vocab_en, vocab_ru)
test_dataset = TranslationDataset(en_test, ru_test, vocab_en, vocab_ru)

# Параметры
BATCH_SIZE = 32

# Создаем загрузчики данных
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

print(f'Количество батчей в train: {len(train_loader)}')
print(f'Количество батчей в val: {len(val_loader)}')
print(f'Количество батчей в test: {len(test_loader)}')

## 4. Модель Transformer

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TranslationTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, nhead=8, 
                 num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, 
                 dropout=0.1, max_len=100, src_pad_idx=0, tgt_pad_idx=0):
        super().__init__()
        
        self.d_model = d_model
        self.src_vocab_size = src_vocab_size
        self.tgt_vocab_size = tgt_vocab_size
        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx
        
        # Эмбеддинги
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len)
        
        # Трансформер
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        # Выходной слой
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, src, tgt):
        # Создаем маски ДО эмбеддингов (когда src и tgt имеют размерность [batch_size, seq_len])
        src_key_padding_mask = (src == self.src_pad_idx)  # [batch_size, src_len]
        tgt_key_padding_mask = (tgt == self.tgt_pad_idx)  # [batch_size, tgt_len]
        
        # Создаем causal mask для decoder
        tgt_len = tgt.size(1)
        causal_mask = torch.triu(torch.ones((tgt_len, tgt_len), device=tgt.device), diagonal=1).bool()
        
        # Эмбеддинги и позиционное кодирование
        src = self.src_embedding(src) * math.sqrt(self.d_model)
        src = self.positional_encoding(src)
        
        tgt = self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        tgt = self.positional_encoding(tgt)
        
        # Прямой проход через трансформер
        output = self.transformer(
            src, tgt,
            tgt_mask=causal_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        
        # Выходной слой
        output = self.fc_out(output)
        
        return output

In [ ]:
# Параметры модели
SRC_VOCAB_SIZE = len(vocab_en)
TGT_VOCAB_SIZE = len(vocab_ru)
SRC_PAD_IDX = vocab_en.stoi["<PAD>"]
TGT_PAD_IDX = vocab_ru.stoi["<PAD>"]
D_MODEL = 256  # Уменьшено для скорости обучения
NHEAD = 4
NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3
DIM_FEEDFORWARD = 512
DROPOUT = 0.1
MAX_LEN = 64  # Установлено с учетом max_length=30 + <SOS>, <EOS>

model = TranslationTransformer(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_len=MAX_LEN,
    src_pad_idx=SRC_PAD_IDX,
    tgt_pad_idx=TGT_PAD_IDX
)

model = model.to(DEVICE)

# Считаем параметры
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Модель создана!')
print(f'Общее количество параметров: {total_params:,}')
print(f'Обучаемых параметров: {trainable_params:,}')

## 5. Обучение модели

In [ ]:
# Параметры обучения
LEARNING_RATE = 0.0001
NUM_EPOCHS = 100
CLIP = 1.0

# Функция потерь (игнорируем PAD)
criterion = nn.CrossEntropyLoss(ignore_index=vocab_ru.stoi["<PAD>"])

# Оптимизатор
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Планировщик скорости обучения
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

print('Параметры обучения:')
print(f'  Learning Rate: {LEARNING_RATE}')
print(f'  Epochs: {NUM_EPOCHS}')
print(f'  Batch Size: {BATCH_SIZE}')

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device, clip):
    model.train()
    epoch_loss = 0
    
    for batch in tqdm(dataloader, desc='Training'):
        src = batch['en'].to(device)
        tgt = batch['ru'].to(device)
        
        # Для обучения: tgt_input - это всё кроме последнего токена
        # tgt_output - это всё кроме первого токена (пропускаем <SOS>)
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        
        optimizer.zero_grad()
        
        # Прямой проход
        output = model(src, tgt_input)
        
        # output: [batch_size, tgt_len-1, vocab_size]
        # tgt_output: [batch_size, tgt_len-1]
        
        # Переставляем размерности для CrossEntropyLoss
        output = output.reshape(-1, output.size(-1))
        tgt_output = tgt_output.reshape(-1)
        
        # Вычисляем потерю
        loss = criterion(output, tgt_output)
        
        # Обратный проход
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        optimizer.step()
        
        epoch_loss += loss.item()
    
    return epoch_loss / len(dataloader)


def evaluate(model, dataloader, criterion, device):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            src = batch['en'].to(device)
            tgt = batch['ru'].to(device)
            
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]
            
            output = model(src, tgt_input)
            
            output = output.reshape(-1, output.size(-1))
            tgt_output = tgt_output.reshape(-1)
            
            loss = criterion(output, tgt_output)
            
            epoch_loss += loss.item()
    
    return epoch_loss / len(dataloader)

In [ ]:
# Списки для хранения истории обучения
train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}')
    print('-' * 50)
    
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE, CLIP)
    val_loss = evaluate(model, val_loader, criterion, DEVICE)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    scheduler.step(val_loss)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.6f}')
    
    # Сохраняем лучшую модель
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, 'best_transformer.pt')
        print(f'Сохранена лучшая модель (val_loss: {val_loss:.4f})')

## 6. Визуализация результатов обучения

In [ ]:
# График потерь
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('График потерь')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('График потерь (логарифмическая шкала)')
plt.yscale('log')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('loss_plots.png', dpi=150)
plt.show()

print(f'\nФинальные результаты:')
print(f'  Train Loss: {train_losses[-1]:.4f}')
print(f'  Val Loss: {val_losses[-1]:.4f}')
print(f'  Best Val Loss: {best_val_loss:.4f}')

## 7. Функция генерации перевода

In [ ]:
def translate(model, sentence, vocab_src, vocab_tgt, device, max_length=50):
    """Перевести одно предложение"""
    model.eval()
    
    with torch.no_grad():
        # Токенизируем исходное предложение
        src_indices = vocab_src.numericalize(sentence)
        src_indices = [vocab_src.stoi["<SOS>"]] + src_indices + [vocab_src.stoi["<EOS>"]]
        src = torch.tensor([src_indices], dtype=torch.long).to(device)
        
        # Начинаем с токена <SOS>
        tgt_indices = [vocab_tgt.stoi["<SOS>"]]
        tgt = torch.tensor([tgt_indices], dtype=torch.long).to(device)
        
        for _ in range(max_length):
            # Получаем предсказание
            output = model(src, tgt)
            
            # Берем последний токен
            output = output[:, -1, :]  # [1, vocab_size]
            
            # Получаем предсказанный токен
            pred_token = output.argmax(-1).item()
            
            # Если <EOS>, заканчиваем
            if pred_token == vocab_tgt.stoi["<EOS>"]:
                break
            
            # Добавляем предсказанный токен
            tgt_indices.append(pred_token)
            tgt = torch.tensor([tgt_indices], dtype=torch.long).to(device)
    
    # Конвертируем индексы обратно в текст
    translated_tokens = [vocab_tgt.itos[idx] for idx in tgt_indices[1:]]  # Пропускаем <SOS>
    
    # Удаляем <EOS> если есть
    if "<EOS>" in translated_tokens:
        translated_tokens = translated_tokens[:translated_tokens.index("<EOS>")]
    
    # Собираем текст
    translated_text = ' '.join(translated_tokens)
    
    # Убираем лишние пробелы перед пунктуацией
    translated_text = re.sub(r'\s+([.,!?;:])', r'\1', translated_text)
    
    return translated_text

# Загружаем лучшую модель
checkpoint = torch.load('best_transformer.pt', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print('Загружена лучшая модель')

## 8. Примеры переводов с разной сложностью

In [ ]:
# Примеры разной сложности
test_sentences = {
    'Простые': [
        "Hello world",
        "How are you?",
        "Good morning",
        "Thank you",
        "Goodbye"
    ],
    'Средние': [
        "The weather is nice today",
        "I want to go to the cinema",
        "She is reading a book",
        "We are very happy",
        "The children play in the garden"
    ],
    'Сложные': [
        "The quick brown fox jumps over the lazy dog",
        "I have been studying Russian for three years now",
        "She speaks three different languages fluently",
        "The beautiful flowers in the garden attract many bees",
        "I would like to invite you to my birthday party next week"
    ]
}

print('=' * 80)
print('ПРИМЕРЫ ПЕРЕВОДОВ РАЗНОЙ СЛОЖНОСТИ')
print('=' * 80)

for difficulty, sentences in test_sentences.items():
    print(f'\n{difficulty}:')
    print('-' * 60)
    
    for sent in sentences:
        translation = translate(model, sent, vocab_en, vocab_ru, DEVICE)
        print(f'EN: {sent}')
        print(f'RU: {translation}')
        print()

## 9. 20 Примеров из тестовой выборки

In [ ]:
print('=' * 80)
print('20 ПРИМЕРОВ ИЗ ТЕСТОВОЙ ВЫБОРКИ')
print('=' * 80)

# Выбираем 20 случайных примеров из тестовой выборки
num_examples = min(20, len(test_dataset))
indices = random.sample(range(len(test_dataset)), num_examples)

for i, idx in enumerate(indices, 1):
    en_sent = en_test[idx]
    ru_sent = ru_test[idx]
    
    # Получаем перевод
    translation = translate(model, en_sent, vocab_en, vocab_ru, DEVICE)
    
    print(f'\nПример {i}:')
    print(f'  Оригинал (EN): {en_sent}')
    print(f'  Эталон (RU):   {ru_sent}')
    print(f'  Перевод (RU):  {translation}')

## 10. Сочиненные примеры для демонстрации

In [ ]:
# Сочиненные примеры для проверки на практике
custom_examples = [
    # Простые
    "Hi!",
    "I am fine",
    "What is it?",
    "Yes, of course",
    "No, thank you",
    
    # Средние
    "I live in a big city",
    "She works at a hospital",
    "They go to school every day",
    "The cat sleeps on the sofa",
    "I drink coffee in the morning",
    
    # Более сложные
    "My friend and I like to watch movies",
    "The teacher explains the lesson to the students",
    "I want to buy a new computer for my work",
    "The beautiful sunset makes me happy",
    "She helps her mother with the housework",
    
    # Комбинированные
    "I am tired but I want to finish my work",
    "The train arrives late and I am worried",
    "We eat dinner together and talk about our day",
    "He studies hard and gets good grades",
    "The weather is cold so I wear a warm coat"
]

print('=' * 80)
print('СОЧИНЕННЫЕ ПРИМЕРЫ ДЛЯ ДЕМОНСТРАЦИИ НА ПРАКТИКЕ')
print('=' * 80)

for i, sentence in enumerate(custom_examples, 1):
    translation = translate(model, sentence, vocab_en, vocab_ru, DEVICE)
    print(f'{i:2d}. EN: {sentence}')
    print(f'    RU: {translation}')
    print()

## 11. Интерактивная проверка (для проверяющих)

In [ ]:
# Функция для интерактивного перевода
def interactive_translate(text):
    """Перевести текст и вернуть результат"""
    if not text.strip():
        return "Пожалуйста, введите текст для перевода"
    
    translation = translate(model, text, vocab_en, vocab_ru, DEVICE)
    return translation

# Пример использования:
user_input = input("Введите английский текст для перевода: ")
if user_input:
    result = interactive_translate(user_input)
    print(f'Перевод: {result}')

## 12. Сохранение результатов и модели

In [ ]:
# Сохраняем модель и словари
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_en_stoi': vocab_en.stoi,
    'vocab_en_itos': vocab_en.itos,
    'vocab_ru_stoi': vocab_ru.stoi,
    'vocab_ru_itos': vocab_ru.itos,
    'model_config': {
        'src_vocab_size': SRC_VOCAB_SIZE,
        'tgt_vocab_size': TGT_VOCAB_SIZE,
        'd_model': D_MODEL,
        'nhead': NHEAD,
        'num_encoder_layers': NUM_ENCODER_LAYERS,
        'num_decoder_layers': NUM_DECODER_LAYERS,
        'dim_feedforward': DIM_FEEDFORWARD,
        'dropout': DROPOUT
    }
}, 'transformer_translation_model.pt')

print('Модель и словари сохранены в файл transformer_translation_model.pt')
print('Графики потерь сохранены в файл loss_plots.png')

## 13. Функция для загрузки модели (для демонстрации на практике)

In [ ]:
def load_model_for_inference(checkpoint_path='transformer_translation_model.pt'):
    """Загрузить модель для инференса"""
    # Загружаем чекпоинт
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    
    # Восстанавливаем словари
    global vocab_en, vocab_ru
    vocab_en = Vocabulary()
    vocab_en.itos = checkpoint['vocab_en_itos']
    vocab_en.stoi = checkpoint['vocab_en_stoi']
    
    vocab_ru = Vocabulary()
    vocab_ru.itos = checkpoint['vocab_ru_itos']
    vocab_ru.stoi = checkpoint['vocab_ru_stoi']
    
    # Создаем модель с той же конфигурацией
    config = checkpoint['model_config']
    model = TranslationTransformer(
        src_vocab_size=config['src_vocab_size'],
        tgt_vocab_size=config['tgt_vocab_size'],
        d_model=config['d_model'],
        nhead=config['nhead'],
        num_encoder_layers=config['num_encoder_layers'],
        num_decoder_layers=config['num_decoder_layers'],
        dim_feedforward=config['dim_feedforward'],
        dropout=config['dropout'],
        src_pad_idx=vocab_en.stoi["<PAD>"],
        tgt_pad_idx=vocab_ru.stoi["<PAD>"]
    )
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(DEVICE)
    model.eval()
    
    return model

# Пример использования функции загрузки:
# loaded_model = load_model_for_inference()
# translation = translate(loaded_model, "Hello world", vocab_en, vocab_ru, DEVICE)
# print(translation)

print('Функция load_model_for_inference() готова для использования')
print('\nИнструкция для запуска на практике:')
print('1. Загрузите ноутбук и все зависимости')
print('2. Выполните все ячейки до ячейки с функцией translate')
print('3. Или используйте функцию load_model_for_inference() для загрузки сохраненной модели')
print('4. Используйте функцию translate() или interactive_translate() для перевода')

---

## Итоговая сводка

В этом ноутбуке мы:

1. **Создали словари** для английского и русского языков
2. **Реализовали модель Transformer** на основе `torch.nn.Transformer`
3. **Обучили модель** на задаче перевода
4. **Построили графики потерь** (train/val)
5. **Показали примеры переводов** разной сложности
6. **Подготовили 20 примеров** из тестовой выборки
7. **Создали сочиненные примеры** для демонстрации на практике

### Для проверки на практике:

```python
# Пример 1: Простой перевод
print(translate(model, "Hello world", vocab_en, vocab_ru, DEVICE))

# Пример 2: Более сложный перевод
print(translate(model, "The weather is nice today", vocab_en, vocab_ru, DEVICE))

# Пример 3: Интерактивный ввод
text = input("Введите текст: ")
print(translate(model, text, vocab_en, vocab_ru, DEVICE))
```